# K-Means 聚类通用模板 · Python 3.11 · 美赛直接复用

**适用**：样本分群 / 区域分类 / 模式识别 / 画像构建，属于无监督学习（不需要标签列）。

## 模型与论文可引用的公式

- 目标：最小化簇内平方和 $SSE=\sum_{k=1}^{K}\sum_{i\in C_k}\|x_i-\mu_k\|^2$。
  Lloyd 算法交替执行「分配到最近中心」与「重算中心为簇均值」，必收敛到局部最优。
- **K-Means++** 初始化：让新中心以正比于 $D(x)^2$ 的概率选取，降低陷入坏局部最优的概率；
  `n_init=10` 再做多次初始化取 SSE 最小解。
- **必须先标准化**：欧氏距离对量纲敏感，未标准化时数值范围大的变量会主导距离。
- 选 K 的四项内部指标：SSE 肘部（拐点）、**轮廓系数**
  $s(i)=\dfrac{b(i)-a(i)}{\max\{a(i),b(i)\}}$（越大越好）、
  Calinski-Harabasz（越大越好）、Davies-Bouldin（越小越好）。
- **稳健性**：不同随机种子重复拟合，用调整兰德指数 ARI 衡量结果是否稳定；
  再与 Ward 层次聚类对照，避免「只有一种算法支持这个分群」。
- **局限**：假设簇近似球形、大小相近；对离群点敏感（本模板先做 1.5IQR 缩尾）；
  类别变量需独热编码后再参与距离计算。

> 依赖：`numpy pandas scipy scikit-learn matplotlib openpyxl`。本 Notebook 自包含。

In [2]:
# ============================================================

# 0. 环境准备（Python 3.11）

# 本 Notebook 自包含：不 import 其他 Notebook，换题目直接整份复制走

# ============================================================

import sys

import warnings

import platform

from pathlib import Path

from dataclasses import dataclass, field, fields, replace



import numpy as np

import pandas as pd

import matplotlib

import matplotlib.pyplot as plt

from scipy import stats

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster



from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.decomposition import PCA as SkPCA

from sklearn.cluster import KMeans

from sklearn.linear_model import LogisticRegression, LinearRegression

from sklearn.metrics import (

    silhouette_score, silhouette_samples,

    calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score,

    accuracy_score, precision_score, recall_score, f1_score,

    roc_auc_score, roc_curve, precision_recall_curve,

    confusion_matrix, classification_report, ConfusionMatrixDisplay,

)

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

from sklearn.base import clone

from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype



if sys.version_info[:2] < (3, 11):

    print(f"[警告] 当前 Python {platform.python_version()}，本模板按 3.11 编写。")

warnings.filterwarnings("ignore", category=FutureWarning)

warnings.filterwarnings("ignore", category=UserWarning)



plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "SimSun", "Arial Unicode MS"]

plt.rcParams["axes.unicode_minus"] = False

plt.rcParams["figure.dpi"] = 110

plt.rcParams["axes.grid"] = True

plt.rcParams["grid.alpha"] = 0.3

pd.set_option("display.unicode.east_asian_width", True)

pd.set_option("display.max_columns", 40)

pd.set_option("display.width", 160)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")



import sklearn

print("Python", platform.python_version(), "| numpy", np.__version__,

      "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

Python 3.11.9 | numpy 2.4.6 | pandas 3.1.0 | sklearn 1.9.1


### 换一份数据，怎么复用？（照抄即可）



1. 把新文件放进 `导入数据/`（`.csv` / `.xlsx` / `.xls` 都支持）；

2. 在 Notebook 末尾新建一格，只写一行：



```python

bundle = run_on("你的文件名.csv", target=None, tag="本次名字")

```



3. `Run All`。结果自动写进 `输出数据/`，文件名形如

`KMeans_示例_结果汇总.xlsx`（前缀固定，中间是 tag）：CSV 明细 + 多 Sheet Excel + 图片。



**数据格式**：一行一个样本、一列一个指标，中文列名没问题。BOM、前后空格、

文本数字、`999` 哨兵、重复行、缺失值、标准化、文字类别转独热编码全部自动处理。



**只有 3 个值要填**：文件名、`target`（`填 None（聚类不需要标签）；若有一列类别想做簇×类别对照也可以填`）、`tag`（换数据就改名，否则覆盖上一份）。

其余参数（`id_cols` / `features` / `na_markers` / `scale` / `k_max` 等）只在需要时改。

`run_on` 也接受任意 Config 字段作关键字参数，例如 `make_plots=False, k_max=10`。



> 输出文件名固定、同名覆盖：再跑 10 次也不会多出文件；换了 `tag` 才会新增一组。

> 想跳过末尾示例就不动 `RUN_EXAMPLES=False`；想一张图都不生成用 `make_plots=False`；

> 清旧结果用 `clean_outputs("旧tag")`。**重新打开 Notebook 才会读到磁盘上的最新版。**

In [4]:
# ============================================================

# 1. 路径 + 配置区（换数据只改这里，或直接用后面的 run_on 一行搞定）

# ============================================================

PREFIX = "KMeans"          # 本 Notebook 所有输出文件的固定前缀



def find_project_root(marker="导入数据"):

    """从当前目录向上找含 导入数据/ 的项目根，兼容从子目录或根目录启动内核。"""

    for base in [Path.cwd(), *Path.cwd().parents]:

        if (base / marker).is_dir():

            return base

    return Path.cwd()



ROOT = find_project_root()

DATA_DIR = ROOT / "导入数据" if (ROOT / "导入数据").is_dir() else ROOT

OUT_DIR = ROOT / "输出数据" if (ROOT / "输出数据").is_dir() else ROOT

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("项目根目录 :", ROOT)

print("输出目录   :", OUT_DIR)





@dataclass

class Config:

    """所有建模开关集中在这里；标 ★ 的是换数据时通常要改的。"""

    data_path: object = None        # ★ 数据文件；None = 自动找 导入数据/ 里第一个表格

    sheet_name: object = 0          # Excel 工作表（名字或序号）

    id_cols: tuple = ("date",)      # ★ 编号/日期等不进模型的列

    target: object = None           # ★ 有监督标签列；None = 无监督（KMeans/PCA 不填）

    features: object = None         # 参与建模的列；None = 自动推断

    tag: str = "demo"               # ★ 输出文件名前缀，换数据记得改名

    make_plots: bool = True         # False = 一张图都不生成（连画布都不创建）



    # ---- 读入与清洗 ----

    na_markers: tuple = (999, 999.0, -999, "未填", "NA", "N/A", "null", "未知")

    drop_duplicates: bool = True

    drop_high_missing: bool = True

    missing_threshold: float = 0.50  # 缺失率超过它就不进模型



    # ---- 预处理 ----

    impute_num: str = "median"      # median / mean / zero

    impute_cat: str = "mode"        # mode / unknown

    add_missing_indicator: bool = True

    outlier: str = "winsor_iqr"     # winsor_iqr / winsor_3sigma / none

    scale: str = "standard"         # standard / minmax / robust / none

    max_onehot_cardinality: int = 30

    onehot_drop_first: bool = False

    random_state: int = 42          # 全局随机种子，保证结果可复现


    # ---- K-Means 专有 ----
    k_min: int = 2                 # 尝试的最小簇数
    k_max: int = 8                 # 尝试的最大簇数
    n_init: int = 10               # 每个 K 的初始化次数
    stability_seeds: int = 5       # 稳健性检验的随机种子个数




CONFIG = Config(data_path=DATA_DIR / "mcm_final_raw.csv", target=None, tag="示例")

RUN_EXAMPLES = False                # True 才会执行末尾的可选示例（多跑一组数据、多产一组文件）





def save_fig(fig, name, cfg):

    """统一保存图片：文件名 = {PREFIX}_{tag}_{图名}.png，同名覆盖；用完关闭画布。"""

    if not getattr(cfg, "make_plots", True):

        plt.close(fig)

        return None

    path = OUT_DIR / f"{PREFIX}_{cfg.tag}_{name}.png"

    fig.savefig(path, dpi=150, bbox_inches="tight")

    plt.show()

    plt.close(fig)                  # 释放 figure，反复 Run All 不会堆积内存

    print(f"[图片] {path.name}（同名覆盖）")

    return path





def clean_outputs(tag=None):

    """清理某个 tag 遗留的产物（只匹配本模板前缀，不碰其他文件）。"""

    tag = CONFIG.tag if tag is None else tag

    files = sorted(OUT_DIR.glob(f"{PREFIX}_{tag}_*"))

    for f in files:

        f.unlink()

    print(f"已清理 tag='{tag}' 的 {len(files)} 个文件")

    return files





print("配置：", CONFIG)

print("运行末尾示例：", RUN_EXAMPLES)

项目根目录 : D:\git项目
输出目录   : D:\git项目\输出数据
配置： Config(data_path=WindowsPath('D:/git项目/导入数据/mcm_final_raw.csv'), sheet_name=0, id_cols=('date',), target=None, features=None, tag='示例', make_plots=True, na_markers=(999, 999.0, -999, '未填', 'NA', 'N/A', 'null', '未知'), drop_duplicates=True, drop_high_missing=True, missing_threshold=0.5, impute_num='median', impute_cat='mode', add_missing_indicator=True, outlier='winsor_iqr', scale='standard', max_onehot_cardinality=30, onehot_drop_first=False, random_state=42, k_min=2, k_max=8, n_init=10, stability_seeds=5)
运行末尾示例： False


### 2. 通用数据读取 `load_table`



自动处理美赛附件的常见坑：CSV 编码/BOM、列名与字符串首尾空格、数字被存成文本、

`999` 等哨兵值、日期列、全空行列、重复行，并打印每一步改了什么（论文预处理段落可直接引用）。

In [6]:
TEXT_NA = ["", " ", "  ", "-", "?", "NA", "N/A", "na", "n/a", "null", "NULL",

           "None", "nan", "NaN", "未填", "未知", "缺失", "无"]





def resolve_data_path(cfg):

    """确定数据文件；相对路径按 导入数据/ 解析，没指定就自动挑一个表格。"""

    if cfg.data_path is not None:

        p = Path(cfg.data_path)

        if not p.is_absolute():

            p = (ROOT / p) if (p.parts and p.parts[0] == DATA_DIR.name) else (DATA_DIR / p)

        if not p.exists():

            raise FileNotFoundError(f"找不到数据文件：{p}\n请检查文件名或传入绝对路径。")

        return p

    cand = sorted([p for p in DATA_DIR.iterdir()

                   if p.suffix.lower() in (".csv", ".xlsx", ".xls")])

    if not cand:

        raise FileNotFoundError(f"{DATA_DIR} 里没有 csv/xlsx 文件。")

    print(f"[提示] 自动选用：{cand[0].name}")

    return cand[0]





def load_table(cfg):

    """读入 → 初清洗 → (DataFrame, 元信息)。数值哨兵留到 3.5 步替换以便统计个数。"""

    path = resolve_data_path(cfg)

    na_values = list(dict.fromkeys(TEXT_NA))

    if path.suffix.lower() in (".xlsx", ".xls"):

        df = pd.read_excel(path, sheet_name=cfg.sheet_name, na_values=na_values)

        enc = "Excel(openpyxl)"

    else:

        df, enc = None, None

        for e in ("utf-8-sig", "utf-8", "gbk", "gb18030"):

            try:

                df = pd.read_csv(path, encoding=e, na_values=na_values,

                                 skipinitialspace=True)

                enc = e

                break

            except UnicodeDecodeError:

                continue

        if df is None:

            raise UnicodeDecodeError("all", b"", 0, 1, "常见编码都无法解码该 CSV。")



    n_raw = len(df)

    # 列名：去 BOM / 去空格 / 重名加后缀

    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

    seen, cols = {}, []

    for c in df.columns:

        seen[c] = seen.get(c, 0) + 1

        cols.append(c if seen[c] == 1 else f"{c}_{seen[c]}")

    df.columns = cols

    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all").copy()



    # 字符串列：strip 后把文字型缺失标记置 NaN

    for c in df.columns:

        if not is_numeric_dtype(df[c]) and not is_datetime64_any_dtype(df[c]):

            df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)

            df[c] = df[c].replace({m: np.nan for m in TEXT_NA})



    # 文本数字列转数值；名字带 date/time 且能解析的列转 datetime

    turned, date_cols = [], []

    for c in df.columns:

        if is_numeric_dtype(df[c]) or is_datetime64_any_dtype(df[c]):

            continue

        s = df[c]

        valid = s.notna().sum()

        if valid == 0:

            continue

        conv = pd.to_numeric(s, errors="coerce")

        if conv.notna().sum() >= 0.90 * valid:

            df[c] = conv.astype(float)

            turned.append(c)

        elif any(k in c.lower() for k in ("date", "time", "日期", "时间")):

            dt = pd.to_datetime(s, errors="coerce")

            if dt.notna().sum() >= 0.90 * valid:

                df[c] = dt

                date_cols.append(c)



    # 数值哨兵（999 等）转 NaN，逐列统计，便于论文说明处理依据

    hits = {}

    num_sentinels = [v for v in cfg.na_markers if isinstance(v, (int, float))]

    for c in df.select_dtypes(include="number").columns:

        hit = int(df[c].isin(num_sentinels).sum())

        if hit:

            hits[c] = hit

            df[c] = df[c].replace({v: np.nan for v in num_sentinels})



    n_dup = int(df.duplicated().sum())

    if cfg.drop_duplicates and n_dup:

        df = df.drop_duplicates().reset_index(drop=True)



    meta = dict(path=path, encoding=enc, n_raw=n_raw, n_final=len(df),

                n_duplicates=n_dup, numeric_turned=turned,

                date_cols=date_cols, sentinel_hits=hits)

    print(f"读取成功 : {path.name}（{enc}）")

    print(f"数据规模 : {n_raw} 行 → {len(df)} 行，{df.shape[1]} 列；删重复 {n_dup} 行")

    print(f"文本转数值 : {turned or '无'}；日期列：{date_cols or '无'}；哨兵转缺失：{hits or '无'}")

    return df, meta





df, META = load_table(CONFIG)

print("\n字段类型：")

print(df.dtypes.to_string())

读取成功 : mcm_final_raw.csv（utf-8-sig）
数据规模 : 381 行 → 366 行，8 列；删重复 15 行
文本转数值 : ['pm25']；日期列：['date']；哨兵转缺失：{'traffic_index': 6}

字段类型：
date                datetime64[us]
region                         str
pm25                       float64
temperature                float64
traffic_index              float64
fuel_type                      str
survey_score               float64
household_income           float64


### 3. 数据体检



建模前先看每列的缺失率、唯一值、极值与 IQR 异常点；缺失率过高的列会在预处理阶段自动剔除。

一张综合图（缺失率 + 标准化箱线图）便于写进论文的数据说明部分。

In [8]:
def data_audit(df, cfg, plot=True):

    """逐列体检表；plot=True 且 make_plots=True 时画一张综合图。"""

    rows = []

    for c in df.columns:

        s = df[c]

        miss = int(s.isna().sum())

        rec = dict(列名=c, 类型=str(s.dtype), 缺失数=miss,

                   缺失率=round(miss / len(df), 4), 唯一值=int(s.nunique(dropna=True)))

        if is_numeric_dtype(s):

            q1, q3 = s.quantile(0.25), s.quantile(0.75)

            iqr = q3 - q1

            rec.update(最小值=round(float(s.min()), 3) if s.notna().any() else np.nan,

                       中位数=round(float(s.median()), 3) if s.notna().any() else np.nan,

                       最大值=round(float(s.max()), 3) if s.notna().any() else np.nan,

                       IQR异常点=int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()),

                       建议=("剔除(缺测过多)" if (cfg.drop_high_missing and

                                             miss / len(df) > cfg.missing_threshold)

                             else ("常数列,剔除" if s.nunique(dropna=True) <= 1 else "保留")))

        else:

            rec.update(最小值="", 中位数="", 最大值="", IQR异常点=0,

                       建议=("日期/标识列,不建模" if is_datetime64_any_dtype(s)

                             else "独热编码" if s.nunique(dropna=True) <= cfg.max_onehot_cardinality

                             else "高基数,剔除"))

        rows.append(rec)

    audit = pd.DataFrame(rows)

    print("===== 数据体检表 =====")

    print(audit.to_string(index=False))



    if plot and cfg.make_plots:

        num = [c for c in df.columns if is_numeric_dtype(df[c])]

        miss_tbl = audit[audit["缺失率"] > 0]

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

        if len(miss_tbl):

            axes[0].barh(miss_tbl["列名"], miss_tbl["缺失率"] * 100, color="#d95f5f")

            axes[0].axvline(cfg.missing_threshold * 100, ls="--", color="#444")

        axes[0].set_title("各列缺失率 (%)"); axes[0].set_xlabel("%")

        if num:

            z = df[num].apply(lambda s: (s - s.mean()) / (s.std() or 1))

            axes[1].boxplot([z[c].dropna() for c in num])

            axes[1].set_xticks(range(1, len(num) + 1))

            axes[1].set_xticklabels(num, rotation=30, ha="right")

            axes[1].set_title("数值列分布（标准化后）")

        fig.tight_layout()

        save_fig(fig, "数据体检", cfg)

    return audit





audit = data_audit(df, CONFIG)

===== 数据体检表 =====
            列名           类型  缺失数  缺失率  唯一值     最小值     中位数     最大值  IQR异常点               建议
            date datetime64[us]       0  0.0000      90                                           0 日期/标识列,不建模
          region            str       0  0.0000       4                                           0           独热编码
            pm25        float64      18  0.0492     344    18.6900    50.4260    93.2070          1               保留
     temperature        float64       5  0.0137     356    11.8600    23.2560    30.6550          0               保留
   traffic_index        float64      15  0.0410     351    43.7890    98.9720   150.0030          4               保留
       fuel_type            str       8  0.0219       4                                           0           独热编码
    survey_score        float64      18  0.0492     342    44.0990    69.7360    99.6440          3               保留
household_income        float64     318  0.8689      42 3,115.6080 5,240.4430 9,9

### 4. 可复用预处理器 `TabularPreprocessor`



`fit` 只在训练数据上统计中位数/众数、缩尾边界、标准化参数、类别清单，

`transform` 对新数据用**同一套参数**，避免数据泄漏。自动剔除标签列、日期列、

高缺失列、常数列、高基数文本列；数值列填补 → 缩尾 → 标准化，文字列众数填补 → 独热编码。

In [10]:
class TabularPreprocessor:

    """任意表格 → 数值特征矩阵 X（保留可读列名），支持 fit/transform 复用。"""



    def __init__(self, cfg):

        self.cfg = cfg

        self.numeric_cols_, self.categorical_cols_ = [], []

        self.indicator_cols_, self.dropped_cols_ = [], {}

        self.fill_num_, self.fill_cat_, self.bounds_ = {}, {}, {}

        self.center_, self.scale_, self.categories_ = {}, {}, {}

        self.feature_names_ = []



    def _pick_columns(self, df):

        excluded = set(self.cfg.id_cols or ())

        if self.cfg.target is not None:

            excluded.add(self.cfg.target)

        for c in df.columns:

            if is_datetime64_any_dtype(df[c]) or any(

                    k in str(c).lower() for k in ("date", "time", "日期", "时间")):

                excluded.add(c)

        cols = [c for c in df.columns if c not in excluded]

        if self.cfg.features is not None:

            cols = [c for c in self.cfg.features if c in df.columns]

        for c in cols:

            s = df[c]

            r = s.isna().mean()

            if r == 1:

                self.dropped_cols_[c] = "全空列"; continue

            if self.cfg.drop_high_missing and r > self.cfg.missing_threshold:

                self.dropped_cols_[c] = f"缺失率 {r:.1%}"; continue

            if s.nunique(dropna=True) <= 1:

                self.dropped_cols_[c] = "常数列"; continue

            if is_numeric_dtype(s):

                self.numeric_cols_.append(c)

            elif s.nunique(dropna=True) > self.cfg.max_onehot_cardinality:

                self.dropped_cols_[c] = "高基数文本列"

            else:

                self.categorical_cols_.append(c)

        if not self.numeric_cols_ and not self.categorical_cols_:

            raise ValueError("没有可用特征，请检查 features / missing_threshold。")



    def fit(self, df):

        self._pick_columns(df)

        cfg = self.cfg

        for c in self.numeric_cols_:

            s = pd.to_numeric(df[c], errors="coerce").astype(float)

            fill = (float(s.mean()) if cfg.impute_num == "mean" else

                    0.0 if cfg.impute_num == "zero" else float(s.median()))

            self.fill_num_[c] = fill if np.isfinite(fill) else 0.0

            if s.isna().any() and cfg.add_missing_indicator and f"{c}_missing" not in df.columns:

                self.indicator_cols_.append(c)

            f = s.fillna(self.fill_num_[c])

            if cfg.outlier == "winsor_iqr":

                q1, q3 = f.quantile(0.25), f.quantile(0.75)

                i = q3 - q1

                lo, hi = float(q1 - 1.5 * i), float(q3 + 1.5 * i)

            elif cfg.outlier == "winsor_3sigma":

                mu, sd = float(f.mean()), float(f.std())

                lo, hi = mu - 3 * sd, mu + 3 * sd

            else:

                lo, hi = -np.inf, np.inf

            self.bounds_[c] = (lo, hi)

            z = f.clip(lo, hi)

            if cfg.scale == "minmax":

                self.center_[c], self.scale_[c] = float(z.min()), float(z.max() - z.min())

            elif cfg.scale == "robust":

                self.center_[c] = float(z.median())

                self.scale_[c] = float(z.quantile(0.75) - z.quantile(0.25))

            elif cfg.scale == "standard":

                self.center_[c], self.scale_[c] = float(z.mean()), float(z.std())

            else:

                self.center_[c], self.scale_[c] = 0.0, 1.0

            if not np.isfinite(self.scale_[c]) or self.scale_[c] == 0:

                self.scale_[c] = 1.0

        for c in self.categorical_cols_:

            s = df[c]

            if cfg.impute_cat == "unknown":

                self.fill_cat_[c] = "未知"

            else:

                m = s.mode(dropna=True)

                self.fill_cat_[c] = str(m.iat[0]) if len(m) else "未知"

            cats = sorted({str(v).strip() for v in s.dropna().unique()})

            if cfg.onehot_drop_first and len(cats) > 1:

                cats = cats[1:]

            self.categories_[c] = cats

        names = list(self.numeric_cols_) + [f"{c}_missing" for c in self.indicator_cols_]

        for c in self.categorical_cols_:

            names += [f"{c}={v}" for v in self.categories_[c]]

        seen, out = {}, []

        for n in names:

            seen[n] = seen.get(n, 0) + 1

            out.append(n if seen[n] == 1 else f"{n}_{seen[n]}")

        self.feature_names_ = out

        return self



    def transform(self, df):

        miss = [c for c in self.numeric_cols_ + self.categorical_cols_ if c not in df.columns]

        if miss:

            raise KeyError(f"新数据缺少建模所需列：{miss}")

        blocks = []

        for c in self.numeric_cols_:

            s = pd.to_numeric(df[c], errors="coerce").astype(float).fillna(self.fill_num_[c])

            lo, hi = self.bounds_[c]

            blocks.append(((s.clip(lo, hi) - self.center_[c]) / self.scale_[c]).to_numpy(float))

        for c in self.indicator_cols_:

            blocks.append(df[c].isna().to_numpy(float))

        for c in self.categorical_cols_:

            s = df[c].map(lambda v: str(v).strip() if pd.notna(v) else self.fill_cat_[c])

            for v in self.categories_[c]:

                blocks.append((s == v).to_numpy(float))

        return pd.DataFrame(np.column_stack(blocks), columns=self.feature_names_, index=df.index)



    def fit_transform(self, df):

        return self.fit(df).transform(df)





pre = TabularPreprocessor(CONFIG).fit(df)

X = pre.transform(df)

print("数值列 :", pre.numeric_cols_)

print("类别列 :", pre.categorical_cols_)

print("剔除列 :", pre.dropped_cols_ or "无")

print("特征矩阵 X :", X.shape)

数值列 : ['pm25', 'temperature', 'traffic_index', 'survey_score']
类别列 : ['region', 'fuel_type']
剔除列 : {'household_income': '缺失率 86.9%'}
特征矩阵 X : (366, 16)


### 5. 选 K、K-Means 拟合、簇画像与可视化

四项内部指标自动选 K，随后给出 PCA 散点、轮廓图、簇画像热图。

In [12]:
# ============================================================
# 5. 选 K：肘部法 + 轮廓系数 + CH + DB
# ============================================================
def find_elbow(values):
    """通用拐点检测：归一化曲线到首尾连线距离最大的点（无需 kneed 库）。"""
    y = np.asarray(values, dtype=float)
    n = len(y)
    if n < 3:
        return n // 2
    rng = y.max() - y.min()
    yn = (y - y.min()) / (rng if rng > 0 else 1)
    xn = np.linspace(0, 1, n)
    line = yn[0] + (yn[-1] - yn[0]) * xn
    dist = np.abs(yn - line) / np.sqrt(1 + (yn[-1] - yn[0]) ** 2)
    return int(np.argmax(dist))


def evaluate_k(X, cfg):
    """逐 K 拟合 K-Means，返回指标表 + 轮廓最优 K + 肘部 K。"""
    rows = []
    for k in range(cfg.k_min, min(cfg.k_max, len(X) - 1) + 1):
        km = KMeans(n_clusters=k, n_init=cfg.n_init,
                    random_state=cfg.random_state).fit(X)
        lab = km.labels_
        rows.append(dict(K=k, SSE=km.inertia_,
                         轮廓系数=silhouette_score(X, lab),
                         CH=calinski_harabasz_score(X, lab),
                         DB=davies_bouldin_score(X, lab)))
    tab = pd.DataFrame(rows)
    k_sil = int(tab.loc[tab["轮廓系数"].idxmax(), "K"])
    k_elb = int(tab["K"].iloc[find_elbow(tab["SSE"].to_numpy())])
    print("===== 各簇数的内部评价指标 =====")
    print(tab.to_string(index=False))
    print(f"\n轮廓系数最优 K = {k_sil}；肘部法 K = {k_elb}（不一致时以轮廓为主，结合业务含义）")
    if cfg.make_plots:
        fig, axes = plt.subplots(2, 2, figsize=(11, 7))
        for ax, (col, title, color) in zip(axes.ravel(), [
                ("SSE", "SSE（越小越好，看拐点）", "#3b7dd8"),
                ("轮廓系数", "轮廓系数（越大越好）", "#d95f5f"),
                ("CH", "Calinski-Harabasz（越大越好）", "#4a9d6a"),
                ("DB", "Davies-Bouldin（越小越好）", "#a06ac9")]):
            ax.plot(tab.K, tab[col], "o-", color=color, lw=2)
            ax.axvline(k_sil, ls="--", color="#d95f5f", label=f"轮廓最优 K={k_sil}")
            ax.axvline(k_elb, ls=":", color="#666", label=f"肘部 K={k_elb}")
            ax.set_xlabel("簇数 K"); ax.set_title(title); ax.legend(fontsize=8)
        fig.suptitle("簇数选择：四项内部评价指标", fontsize=13)
        fig.tight_layout(); save_fig(fig, "选K指标", cfg)
    return tab, k_sil, k_elb


def cluster_profile(df, labels, cfg):
    """簇画像：数值列取原始量纲均值、类别列取众数，用于给簇命名和下结论。"""
    tmp = df.copy()
    tmp["cluster"] = labels
    num = [c for c in df.columns if is_numeric_dtype(df[c])]
    cat = [c for c in df.columns if not is_numeric_dtype(df[c])
           and not is_datetime64_any_dtype(df[c]) and c not in (cfg.id_cols or ())]
    parts = []
    if num:
        parts.append(tmp.groupby("cluster")[num].mean().round(3))
    for c in cat:
        parts.append(tmp.groupby("cluster")[c].agg(
            lambda s: s.mode(dropna=True).iat[0] if len(s.mode(dropna=True)) else "未知"
        ).rename(c).to_frame())
    prof = pd.concat(parts, axis=1)
    prof.insert(0, "样本数", tmp.groupby("cluster").size())
    return prof


def plot_clusters(X, labels, centers, cfg):
    """PCA 二维投影画聚类散点（聚类本身仍在高维标准化空间进行）。"""
    if not cfg.make_plots:
        return None
    pca = SkPCA(n_components=2, random_state=cfg.random_state)
    Z = pca.fit_transform(X.to_numpy())
    C = pca.transform(np.asarray(centers))
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for k in sorted(set(labels)):
        m = labels == k
        ax.scatter(Z[m, 0], Z[m, 1], s=22, alpha=0.75, label=f"簇 {k}（n={m.sum()}）")
    ax.scatter(C[:, 0], C[:, 1], marker="X", s=220, c="black",
               edgecolor="white", label="簇中心")
    ax.set_xlabel(f"PC1（{pca.explained_variance_ratio_[0]:.1%}）")
    ax.set_ylabel(f"PC2（{pca.explained_variance_ratio_[1]:.1%}）")
    ax.set_title("K-Means 聚类结果（PCA 二维投影）")
    ax.legend(fontsize=8); fig.tight_layout(); save_fig(fig, "PCA散点", cfg)
    return pca


def plot_silhouette(X, labels, cfg):
    """每个样本一条横线的轮廓图，红虚线为总体平均值。"""
    if not cfg.make_plots:
        return
    v = silhouette_samples(X.to_numpy(), labels)
    fig, ax = plt.subplots(figsize=(7, 5))
    y0 = 0
    for k in sorted(set(labels)):
        s = np.sort(v[labels == k])
        ax.fill_betweenx(np.arange(y0, y0 + len(s)), 0, s, alpha=0.8, label=f"簇 {k}")
        y0 += len(s)
    ax.axvline(float(v.mean()), ls="--", color="red", label=f"平均 {v.mean():.3f}")
    ax.set_xlabel("轮廓系数 s(i)"); ax.set_ylabel("样本（按簇排列）")
    ax.set_title("各簇轮廓系数分布"); ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout(); save_fig(fig, "轮廓图", cfg)


def plot_profile_heatmap(prof, cfg):
    """簇画像热图：每列跨簇做 z-score，颜色显示哪个簇在该指标上偏高/偏低。"""
    if not cfg.make_plots:
        return
    num = prof.select_dtypes(include="number").drop(columns=["样本数"], errors="ignore")
    if num.empty:
        return
    Z = (num - num.mean()) / num.std().replace(0, 1)
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * num.shape[1]), max(2.6, 0.8 * len(Z))))
    im = ax.imshow(Z.to_numpy(), cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    ax.set_xticks(range(num.shape[1]))
    ax.set_xticklabels(num.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(Z)))
    ax.set_yticklabels([f"簇 {i}" for i in Z.index])
    ax.set_title("簇画像热图（红色=相对偏高，蓝色=偏低）")
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="相对均值（标准差单位）")
    fig.tight_layout(); save_fig(fig, "簇画像热图", cfg)


def run_kmeans(df, X, k, cfg):
    km = KMeans(n_clusters=k, n_init=cfg.n_init,
                random_state=cfg.random_state).fit(X)
    out = df.copy()
    out["cluster"] = km.labels_
    prof = cluster_profile(df, km.labels_, cfg)
    pca = plot_clusters(X, km.labels_, km.cluster_centers_, cfg)
    plot_silhouette(X, km.labels_, cfg)
    plot_profile_heatmap(prof, cfg)
    print(f"\nK-Means 完成：K={k}，平均轮廓 {silhouette_score(X, km.labels_):.4f}")
    print("簇样本数：", dict(pd.Series(km.labels_).value_counts().sort_index()))
    print("\n簇画像（原始量纲）：")
    print(prof.to_string())
    return out, km, pca, prof


k_table, K_BEST, K_ELBOW = evaluate_k(X, CONFIG)
CLU_DF, KM, PCA2, PROFILE = run_kmeans(df, X, K_BEST, CONFIG)

===== 各簇数的内部评价指标 =====
 K        SSE  轮廓系数      CH     DB
 2 1,749.9699    0.1362 61.6696 2.3351
 3 1,553.1440    0.1344 57.6480 2.0273
 4 1,422.1795    0.1329 52.9673 1.9127
 5 1,310.3787    0.1348 50.6958 1.7637
 6 1,243.0546    0.1283 46.5343 1.7270
 7 1,181.2073    0.1295 43.8285 1.7451
 8 1,127.7567    0.1216 41.6622 1.8109

轮廓系数最优 K = 2；肘部法 K = 5（不一致时以轮廓为主，结合业务含义）
[图片] KMeans_示例_选K指标.png（同名覆盖）
[图片] KMeans_示例_PCA散点.png（同名覆盖）
[图片] KMeans_示例_轮廓图.png（同名覆盖）
[图片] KMeans_示例_簇画像热图.png（同名覆盖）

K-Means 完成：K=2，平均轮廓 0.1362
簇样本数： {0: np.int64(161), 1: np.int64(205)}

簇画像（原始量纲）：
         样本数    pm25  temperature  traffic_index  survey_score  household_income region fuel_type
cluster                                                                                             
0           161 38.6030      23.0550       101.1180       68.6320        5,310.6890   西部    天然气
1           205 60.9270      22.0460        98.0460       71.8320        5,981.7780   东部    天然气


### 6. 稳健性与敏感性检验

多种子重复拟合看 ARI、换标准化方式看结论是否变、与 Ward 层次聚类对照。

In [14]:
# ============================================================
# 6. 稳健性与敏感性检验（论文里最加分的两块）
# ============================================================
def stability_check(X, k, cfg):
    """多个随机种子重复拟合，两两计算 ARI；越接近 1 说明分群越稳定。"""
    labs = [KMeans(n_clusters=k, n_init=cfg.n_init,
                   random_state=cfg.random_state + s).fit_predict(X)
            for s in range(cfg.stability_seeds)]
    aris = [adjusted_rand_score(labs[i], labs[j])
            for i in range(len(labs)) for j in range(i + 1, len(labs))]
    res = dict(种子数=cfg.stability_seeds, ARI均值=round(float(np.mean(aris)), 4),
               ARI最小值=round(float(np.min(aris)), 4),
               ARI标准差=round(float(np.std(aris)), 4),
               结论="分群稳定" if np.mean(aris) > 0.75 else "存在波动，建议增大 n_init 或换 K")
    print("===== 初始化稳健性 =====")
    print(pd.DataFrame([res]).to_string(index=False))
    return res


def compare_scaling(df, cfg):
    """预处理敏感性：三种标准化方式下各 K 的最佳轮廓系数，检验结论是否依赖预处理选择。"""
    rows = []
    for sc in ("standard", "minmax", "robust"):
        c2 = replace(cfg, scale=sc)              # 其余配置完全不变，只换 scale
        p2 = TabularPreprocessor(c2).fit(df)
        X2 = p2.transform(df)
        best, bk = -1.0, None
        for k in range(cfg.k_min, min(cfg.k_max, 6) + 1):
            lab = KMeans(n_clusters=k, n_init=cfg.n_init,
                         random_state=cfg.random_state).fit_predict(X2)
            s = silhouette_score(X2, lab)
            if s > best:
                best, bk = s, k
        rows.append(dict(标准化方式=sc, 最佳K=bk, 最佳轮廓系数=round(best, 4)))
    tab = pd.DataFrame(rows)
    print("\n===== 预处理敏感性（只换标准化方式）=====")
    print(tab.to_string(index=False))
    return tab


def hierarchical_check(X, k, labels, cfg):
    """Ward 层次聚类作对照，ARI 衡量两种方法的分群一致性（不额外画图）。"""
    Z = linkage(X.to_numpy(), method="ward")
    hl = fcluster(Z, t=k, criterion="maxclust") - 1
    ari = adjusted_rand_score(labels, hl)
    print(f"\nWard 层次聚类 vs K-Means：ARI = {ari:.4f}（>0.6 说明分群结构较稳健）")
    if ari < 0.4:
        print("[提示] 两种算法分群不一致：K-Means 的球形簇假设可能不成立，"
              "论文里应结合轮廓系数谨慎解释，或考虑改用层次聚类 / DBSCAN 的结果。")
    return dict(ARI=round(ari, 4), hier_labels=hl, linkage_matrix=Z)


STAB = stability_check(X, K_BEST, CONFIG)
SENS = compare_scaling(df, CONFIG)
HIER = hierarchical_check(X, K_BEST, KM.labels_, CONFIG)

===== 初始化稳健性 =====
 种子数  ARI均值  ARI最小值  ARI标准差     结论
      5   1.0000     1.0000     0.0000 分群稳定

===== 预处理敏感性（只换标准化方式）=====
标准化方式  最佳K  最佳轮廓系数
  standard      2        0.1362
    minmax      4        0.3301
    robust      4        0.1741

Ward 层次聚类 vs K-Means：ARI = 0.1175（>0.6 说明分群结构较稳健）
[提示] 两种算法分群不一致：K-Means 的球形簇假设可能不成立，论文里应结合轮廓系数谨慎解释，或考虑改用层次聚类 / DBSCAN 的结果。


### 结果怎么拿

`bundle` 装着所有中间对象，常用的几个键见下面各节；同时结果会写成
`输出数据/` 下以 `KMeans` 开头、带 tag 的 `_结果明细.csv`（原表 + 预测/得分）
与 `_结果汇总.xlsx`（多 Sheet）；图片单独存 PNG。
换 tag 会新增一组文件，同 tag 是覆盖。

In [16]:
# ============================================================
# 7. 结果导出 + 新数据分群
# ============================================================
def export_results(bundle, cfg):
    out = bundle["clu_df"].copy()
    out["hier_cluster"] = bundle["hier"]["hier_labels"]
    csv = OUT_DIR / f"{PREFIX}_{cfg.tag}_结果明细.csv"
    out.to_csv(csv, index=False, encoding="utf-8-sig")
    sheets = {"数据体检": bundle["audit"], "选K指标": bundle["k_table"],
              "簇画像": bundle["profile"].reset_index(),
              "初始化稳健性": pd.DataFrame([bundle["stability"]]),
              "预处理敏感性": bundle["sensitivity"]}
    xlsx = OUT_DIR / f"{PREFIX}_{cfg.tag}_结果汇总.xlsx"
    try:
        with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
            for n, t in sheets.items():
                t.to_excel(w, sheet_name=n[:31], index=False)
        print(f"[Excel] {xlsx.name}（{len(sheets)} 个工作表）")
    except ImportError:
        print("[提示] 缺 openpyxl，跳过 Excel；CSV 已导出。")
    print(f"[CSV]   {csv.name}")
    return csv, xlsx


def predict_new(bundle, df_new):
    """新数据分群：复用同一个预处理器与训练好的中心，不会重新拟合。"""
    Xn = bundle["preprocessor"].transform(df_new)
    out = df_new.copy()
    out["cluster_pred"] = bundle["kmeans"].predict(Xn)
    print(f"新数据 {len(out)} 行完成分群：",
          dict(pd.Series(out.cluster_pred).value_counts().sort_index()))
    return out

### 一键入口 `run_pipeline` / `run_on`

`run_on("文件名", target=..., tag=...)` 是最省事的写法：自动定位文件、检查存在性、
校验参数名、创建配置、跑完整流程并返回 `bundle`。想逐项调参就用 `Config + run_pipeline`。

In [18]:
# ============================================================
# 8. 一键流程 + run_on
# ============================================================
def run_pipeline(cfg):
    print("=" * 68)
    print(f"K-Means：{cfg.tag} | 数据 {cfg.data_path}")
    print("=" * 68)
    d, meta = load_table(cfg)
    aud = data_audit(d, cfg)
    p = TabularPreprocessor(cfg).fit(d)
    Xv = p.transform(d)
    kt, kb, ke = evaluate_k(Xv, cfg)
    clu, km, pca, prof = run_kmeans(d, Xv, kb, cfg)
    stab = stability_check(Xv, kb, cfg)
    sens = compare_scaling(d, cfg)
    hier = hierarchical_check(Xv, kb, km.labels_, cfg)

    bundle = dict(cfg=cfg, meta=meta, df=d, audit=aud, preprocessor=p, X=Xv,
                  k_table=kt, k_best=kb, k_elbow=ke, clu_df=clu, kmeans=km,
                  pca=pca, profile=prof, stability=stab, sensitivity=sens, hier=hier)
    export_results(bundle, cfg)
    print("=" * 68)
    print(f"完成：{cfg.tag} | 最优 K={kb}（肘部 K={ke}）| 结果见 {OUT_DIR}")
    print("=" * 68)
    return bundle


def run_on(data_path, target=None, tag="kmeans_demo", **overrides):
    """一行换数据：bundle = run_on("新数据.csv", tag="本次名字")"""
    path = Path(data_path)
    if not path.is_absolute():
        path = (ROOT / path) if (path.parts and path.parts[0] == DATA_DIR.name) else (DATA_DIR / path)
    if not path.exists():
        raise FileNotFoundError(f"找不到数据文件：{path}\n请把文件放进 {DATA_DIR}。")
    valid = {f.name for f in fields(Config)}
    bad = set(overrides) - valid
    if bad:
        raise ValueError(f"未知配置项 {sorted(bad)}；可用字段：{sorted(valid)}")
    params = dict(data_path=path, target=target, tag=tag)
    params.update(overrides)
    # 从当前 CONFIG 继承未显式覆盖的字段（例如逻辑回归默认丢弃独热参照组），
    # 这样「改 CONFIG 就生效」与「用 run_on 一行换数据」两条路径行为一致
    return run_pipeline(replace(CONFIG, **params))

### 主执行（默认运行这一格，就出一套完整结果）

把下面这一行的三个值换成你的文件名、标签列、tag，其余都不用动。

In [20]:
# ============================================================
# 主执行：换题目时只改这一行
# ============================================================

bundle = run_on("mcm_final_raw.csv", tag="示例")
print("bundle 可用键：", sorted(bundle.keys()))

K-Means：示例 | 数据 D:\git项目\导入数据\mcm_final_raw.csv
读取成功 : mcm_final_raw.csv（utf-8-sig）
数据规模 : 381 行 → 366 行，8 列；删重复 15 行
文本转数值 : ['pm25']；日期列：['date']；哨兵转缺失：{'traffic_index': 6}
===== 数据体检表 =====
            列名           类型  缺失数  缺失率  唯一值     最小值     中位数     最大值  IQR异常点               建议
            date datetime64[us]       0  0.0000      90                                           0 日期/标识列,不建模
          region            str       0  0.0000       4                                           0           独热编码
            pm25        float64      18  0.0492     344    18.6900    50.4260    93.2070          1               保留
     temperature        float64       5  0.0137     356    11.8600    23.2560    30.6550          0               保留
   traffic_index        float64      15  0.0410     351    43.7890    98.9720   150.0030          4               保留
       fuel_type            str       8  0.0219       4                                           0           独热编码
    survey_score       

### 可选示例（默认不执行，把 RUN_EXAMPLES 改成 True 才会跑）

演示换数据、换参数、新数据推理；跑起来会多产出一小组文件。

In [22]:
if RUN_EXAMPLES:
    # 示例 1：换清洗后的数据，改用 minmax 标准化、最多试 6 个簇、不生成图片
    b2 = run_on(OUT_DIR / "final_cleaned.csv", tag="清洗数据", make_plots=False,
                k_max=6, scale="minmax")
    # 示例 2：新数据复用训练好的中心
    new_pred = predict_new(bundle, bundle["df"].head(8))
    print(new_pred[["cluster_pred"]].to_string())
    # 示例 3：清掉某一组历史结果
    # clean_outputs("清洗数据")
else:
    print("已跳过可选示例（RUN_EXAMPLES=False）。")

已跳过可选示例（RUN_EXAMPLES=False）。
